<a href="https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/01_build_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · Build the triage agent

You'll build the agent the rest of the lab is about. It reads an incident report from a field
site and decides what happened, how bad it is, and whether the HSE manager needs to hear about
it now.

**New in this lesson:** `create_agent`, tools, structured output, reading a trace.

> **Need a key?** Store your LangSmith API key in Colab Secrets (🔑 in the left sidebar) as
> `LANGSMITH_API_KEY`, with **Notebook access** on. If you haven't yet, run
> **[00 · Setup](https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/00_setup.ipynb)** first.

In [ ]:
# --- snippet:setup v1 ---
import os

try:
    from google.colab import userdata
except ImportError:
    userdata = None

if userdata:
    %pip install -qq --progress-bar off \
      "langchain~=1.4.3" \
      "langchain-openai~=1.6.7" \
      "langsmith~=0.14.4" \
      "agentevals~=0.0.9"

    try:
        os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        raise RuntimeError(
            "Add LANGSMITH_API_KEY in the 🔑 Secrets panel, turn on Notebook access, "
            "then run this cell again."
        ) from None
else:
    from dotenv import load_dotenv

    load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "hse-01-build-agent"

MODEL = "langsmith:openai/gpt-6-luna"
# --- /snippet ---

print("Ready.")

---

## 1. The data

The agent works for **Juniper Basin Energy**, a made-up operator with eight field sites. It has
two pieces of reference data: a registry of the sites, and a policy that defines each severity
level and says when to escalate. Run the cell to load them.

In [ ]:
#@title Site registry and severity policy { display-mode: "form" }
SITES = [
    {"site_id": "CC-03", "name": "Cedar Creek Pad 3", "kind": "well pad", "area": "North"},
    {"site_id": "CC-05", "name": "Cedar Creek Pad 5", "kind": "well pad", "area": "North"},
    {"site_id": "RM-TF", "name": "Red Mesa Tank Farm", "kind": "tank farm", "area": "South"},
    {"site_id": "RM-CS", "name": "Red Mesa Compressor Station", "kind": "compressor station", "area": "South"},
    {"site_id": "WF-GP", "name": "Willow Flats Gas Plant", "kind": "gas plant", "area": "East"},
    {"site_id": "DF-WF", "name": "Dry Fork Water Facility", "kind": "water facility", "area": "East"},
    {"site_id": "PR-PY", "name": "Pine Ridge Pipe Yard", "kind": "pipe yard", "area": "West"},
    {"site_id": "EP-FO", "name": "Eagle Point Field Office", "kind": "office", "area": "West"},
]

SEVERITY_POLICY = """Severity levels
- low: nobody hurt; any spill is under 50 litres and stays contained; no fire; minor damage.
- medium: first aid only; a spill of 50 to 500 litres, or any spill that leaves its containment;
  a small fire put out on site; damage that takes a piece of equipment out of service.
- high: an injury that needs a doctor; a spill over 500 litres; a fire that needs the fire
  department; a gas release.
- critical: a life-threatening injury; a spill that reaches a creek or river; an explosion; the
  site is evacuated.
When a report fits two levels, pick the higher one.

Escalate to the HSE manager when any of these is true:
- anyone was hurt, even first aid only
- there was a fire, even one put out at once
- the severity is high or critical"""

In [ ]:
print(SEVERITY_POLICY)

---

## 2. The tools

A tool is a plain Python function. The model never sees the code, only the name, the arguments
and the docstring, so **the docstring is part of the prompt**.

In [ ]:
import json
from difflib import get_close_matches


def lookup_site(name: str) -> str:
    """Find a site in the registry by name. Returns the closest site."""
    matches = [s for s in SITES if name.casefold() in s["name"].casefold()]
    if len(matches) > 1:
        return f"More than one site matches {name!r}. Be more specific."
    if not matches:
        closest = get_close_matches(name, [s["name"] for s in SITES], n=1, cutoff=0)
        matches = [s for s in SITES if s["name"] == closest[0]]
    return json.dumps(matches[0])


def get_severity_policy() -> str:
    """Get the severity levels and the rules for escalating to the HSE manager."""
    return SEVERITY_POLICY

---

## 3. The output

Every triage result has the same shape. With `response_format`, the agent returns a `Triage`
object instead of free text, so the code downstream, and the evaluators later in the lab, can
rely on its fields.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field


class Triage(BaseModel):
    incident_type: Literal["injury", "spill", "fire", "near_miss", "equipment", "vehicle"]
    severity: Literal["low", "medium", "high", "critical"]
    site: str = Field(description='The site\'s name from the registry, or "unknown"')
    injury: bool = Field(description="Was anyone hurt?")
    escalate: bool = Field(description="Should the HSE manager hear about it now?")
    reason: str = Field(description="One sentence on why this severity")

---

## 4. The prompt and the agent

`create_agent` puts it together: a model, the tools, a system prompt, and the output shape. The
agent calls tools in a loop until it can fill in a `Triage`.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = """You triage HSE (health, safety and environment) incident reports from the field
sites of Juniper Basin Energy.

For each report:
1. Find the site with lookup_site.
2. Classify the incident and set its severity from the reporter's written description.
3. Decide whether to escalate to the HSE manager. If you are unsure about the severity, check
   get_severity_policy."""

agent = create_agent(
    model=MODEL,
    tools=[lookup_site, get_severity_policy],
    system_prompt=SYSTEM_PROMPT,
    response_format=Triage,
    name="hse-triage",
)

---

## 5. Triage a report

`triage` sends one report to the agent, with an optional photo, and returns its `Triage`.

In [ ]:
from langchain.messages import HumanMessage


def triage(report: str, photo: str | None = None) -> Triage:
    """Run the agent on one report. `photo` is an optional base64 JPEG."""
    content = [{"type": "text", "text": report}]
    if photo:
        content.append({"type": "image", "base64": photo, "mime_type": "image/jpeg"})
    result = agent.invoke({"messages": [HumanMessage(content=content)]})
    return result["structured_response"]


triage(
    "Cedar Creek Pad 5. Pump seal leaking into the drip tray, about 10 litres. Contained and "
    "cleaned up."
)

In [ ]:
triage(
    "Red Mesa Compressor Station. A contractor cut a hand on a valve handle. First aid on site, "
    "back to work within the hour."
)

Now write one of your own.

In [ ]:
triage("Write your own incident report here.")

---

## 6. Read the trace

Every call above was traced. Open the link and click the most recent trace. You'll see:

- the **input**: the report the agent got;
- each **model call**, with the full prompt the model saw;
- each **tool call**, with its arguments and what it returned;
- the **output**: the `Triage` the agent settled on.

Check which tools the agent called, and in what order. You'll need that habit next.

In [ ]:
from langsmith import Client

print("Your traces:", Client().read_project(project_name=os.environ["LANGSMITH_PROJECT"]).url)

---

## 7. Optional: chat with it in Studio

Studio gives you a chat window on the agent, with each step laid out as it runs. From Colab it
needs a tunnel to your browser, which some company networks block. **If the link doesn't load,
skip this section.** Nothing later depends on it.

1. Run the cell and open the link it prints.
2. If Studio asks about the domain: **Advanced Settings** → **Allowed Domains**, add
   `*.trycloudflare.com`.
3. Paste a report into the chat.

In [ ]:
# Not on PyPI: pinned to a commit.
%pip install -qq --progress-bar off \
  "langsmith-studio-nb @ git+https://github.com/langchain-samples/langsmith-studio-nb@be33f66fa30972892ccf5c412adac12c965d6bdd"

from langsmith_studio_nb import start_studio

start_studio("agent")

In [ ]:
from langsmith_studio_nb import stop_studio

stop_studio()

---

## 📌 Key takeaways

- `create_agent` is a model, tools, a system prompt and an output shape. `response_format` gives
  you typed fields you can check in code.
- The prompt and the tool docstrings are the agent's instructions. Small wording choices change
  what it does.
- Every run is a trace: the input, each model call, each tool call, the output.
- A few reports that look right don't prove the agent works. Next you'll see what it did after a
  week in production.

## ➡️ Next

**[02 · The hunt](https://colab.research.google.com/github/darieldatoon/hse-agent-workshop/blob/main/notebooks/02_hunt.ipynb)**